model building and testing

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/content/clean_df.csv')
df.head()

,Unnamed: 0,company,country_of_origin,model,number_of_cc,horsepower,torque,transmission_type,drivetrain,seating,price,year,looks,body_type,engine_type,number_of_cylinders
0,0,italy,Italy,rs 660,659.0,100.0,67.0,quickshifter,chain,2,1099000.0,2021,sport,naked,paralleltwin,2
1,1,italy,Italy,tuono 660,659.0,100.0,67.0,quickshifter,chain,2,1199000.0,2021,sport,naked,paralleltwin,2
2,2,italy,Italy,rs 125,124.9,15.0,12.0,manual,chain,2,449000.0,2022,sport,racing,singlecylinder,1
3,3,italy,Italy,shiver 900,896.0,95.0,90.0,manual,shaft,2,1399000.0,2022,adventure,naked,vtwin,2
4,4,italy,Italy,tuono 1100,1077.0,175.0,121.0,manual,shaft,2,1999000.0,2022,adventure,naked,vtwin,2


In [ ]:
df.drop(columns=['country_of_origin','number_of_cylinders','Unnamed: 0'],inplace=True)
df.head()

,company,model,number_of_cc,horsepower,torque,transmission_type,drivetrain,seating,price,year,looks,body_type,engine_type
0,italy,rs 660,659.0,100.0,67.0,quickshifter,chain,2,1099000.0,2021,sport,naked,paralleltwin
1,italy,tuono 660,659.0,100.0,67.0,quickshifter,chain,2,1199000.0,2021,sport,naked,paralleltwin
2,italy,rs 125,124.9,15.0,12.0,manual,chain,2,449000.0,2022,sport,racing,singlecylinder
3,italy,shiver 900,896.0,95.0,90.0,manual,shaft,2,1399000.0,2022,adventure,naked,vtwin
4,italy,tuono 1100,1077.0,175.0,121.0,manual,shaft,2,1999000.0,2022,adventure,naked,vtwin


In [ ]:
imp_features = ['company','horsepower','number_of_cc','torque','transmission_type','drivetrain','seating','looks','body_type','engine_type']

def merge_features(df):
  # Initialize 'tags' column if it doesn't exist, or clear it if it does
  if 'tags' not in df.columns:
      df['tags'] = ''
  else:
      df['tags'] = ''

  features_to_drop = []
  # Filter imp_features to only include columns currently in df
  available_features = [f for f in imp_features if f in df.columns]

  for feature in available_features:

    df['tags'] = df['tags'] + ' ' + df[feature].astype(str)
    features_to_drop.append(feature)

  # Drop the original feature columns that were actually merged, ignoring errors for robustness
  df.drop(columns=features_to_drop, inplace=True, errors='ignore')
  return df

df = merge_features(df)
df.head()

,model,price,year,tags
0,rs 660,1099000.0,2021,100.0 659.0 67.0 quickshifter chain 2 sport n...
1,tuono 660,1199000.0,2021,100.0 659.0 67.0 quickshifter chain 2 sport n...
2,rs 125,449000.0,2022,15.0 124.9 12.0 manual chain 2 sport racing s...
3,shiver 900,1399000.0,2022,95.0 896.0 90.0 manual shaft 2 adventure nake...
4,tuono 1100,1999000.0,2022,175.0 1077.0 121.0 manual shaft 2 adventure n...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectors  = CountVectorizer(max_features=5000,stop_words='english')

from sklearn.metrics.pairwise import cosine_similarity
simailarity = cosine_similarity(vectors.fit_transform(df['tags']).toarray())

In [ ]:
simailarity.shape

(305, 305)

In [ ]:
def get_recomendations(bike_model):
  car_index = df[df['model'] == bike_model].index[0]
  distances = simailarity[car_index]
  bike_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

  for i in bike_list:
    print(df.iloc[i[0]].model , df.iloc[i[0]].price , df.iloc[i[0]].year)


In [ ]:
df.sample(5)

,model,price,year,tags
146,rr 250 4-stroke,1200000.0,2023,38.0 250.0 28.0 sequential chain drive 1 clas...
178,glamour 125,72000.0,2023,10.7 124.7 10.6 mesh chain 1 modern stylish c...
150,mc 250f,1100000.0,2023,37.0 250.0 26.0 speed chain drive 1 modern en...
250,renegade commando,164000.0,2023,25.5 279.5 23.0 speed chain 2 retro cruiser s...
183,ntorq 125,69900.0,2019,9.4 124.8 10.5 cvt automatic 2 fun scooter si...


In [ ]:
get_recomendations('mc 250f')

ec 250f 1150000.0 2023
mc 350f 1300000.0 2023
mc 450f 1450000.0 2023
rr 390 4-stroke 1400000.0 2023
rr 250 4-stroke 1200000.0 2023


In [ ]:
import pickle
pickle.dump(df,open('df.pkl','wb'))
pickle.dump(simailarity,open('simailarity.pkl','wb'))